# Code Summarization via LSTM

**CSCI 455/555 — GenAI for SD, Assignment 2**

This notebook trains an encoder-decoder LSTM to generate natural language summaries for Java methods.

## Data
- **Source:** [CodeXGLUE code-to-text (Java)](https://huggingface.co/datasets/google/code_x_glue_ct_code_to_text) — CodeSearchNet Java subset.
- **Pre-processing:** Each Java method is flattened to one whitespace-normalized line. Each summary is lowercased. Trivially short or boilerplate samples are filtered.
- **Split:** ~50,000 training pairs, 1,000 validation pairs.
- **Tokenization & Embeddings:** CodeT5+ (Salesforce/codet5p-220m) via the provided `get_codet5_embeddings.py` script. Vocab size = 32,100, embedding dim = 768.

## Model
- 2-layer LSTM encoder-decoder with teacher forcing.
- Pretrained CodeT5+ embeddings (fine-tuned during training).
- Early stopping on validation BLEU-1 (patience=3).

## Evaluation
- BLEU-1/2/3/4 (sacrebleu), METEOR (nltk), BERTScore, SIDE.

## 1. Setup & Dependencies

In [ ]:
!pip install -q datasets transformers sentencepiece sacrebleu nltk bert-score tqdm matplotlib
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import numpy as np
import os, csv, random
from tqdm import tqdm
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

## 2. Data Collection & Pre-processing

We mine Java method–summary pairs from the **CodeSearchNet** corpus (via the CodeXGLUE code-to-text dataset on Hugging Face).

Pre-processing steps:
1. Flatten each Java method into a single whitespace-normalized line.
2. Lowercase each summary.
3. Filter out trivially short methods (< 5 tokens) or summaries (< 3 tokens).
4. Write to `.txt` files (one sample per line) for the embedding script.

In [ ]:
# Download and prepare dataset
# This creates train_code.txt, train_summary.txt, val_code.txt, val_summary.txt
!python get_data.py

In [ ]:
# Verify counts
for f in ['train_code.txt', 'train_summary.txt', 'val_code.txt', 'val_summary.txt']:
    with open(f) as fh:
        n = sum(1 for _ in fh)
    print(f'{f}: {n} lines')

# Show a few samples
print('\n--- Sample code ---')
with open('train_code.txt') as f:
    for i, line in enumerate(f):
        if i >= 2: break
        print(line.strip()[:200], '...')

print('\n--- Sample summaries ---')
with open('train_summary.txt') as f:
    for i, line in enumerate(f):
        if i >= 2: break
        print(line.strip())

## 3. Tokenization & Embedding Extraction (CodeT5+)

We use the provided `get_codet5_embeddings.py` to:
- Tokenize all data with the CodeT5+ tokenizer (BPE).
- Extract the pretrained embedding matrix (32,100 × 768).

We choose `max_length=256` for code (Java methods can be long) and `max_length=64` for summaries (natural language, typically short).

In [ ]:
!pip install -q -r provided-files/requirements.txt

In [ ]:
# Tokenize training code and summaries
!python provided-files/get_codet5_embeddings.py --input train_code.txt --output train_code.pt --max_length 256
!python provided-files/get_codet5_embeddings.py --input train_summary.txt --output train_summary.pt --max_length 64

In [ ]:
# Tokenize validation code and summaries
!python provided-files/get_codet5_embeddings.py --input val_code.txt --output val_code.pt --max_length 256
!python provided-files/get_codet5_embeddings.py --input val_summary.txt --output val_summary.pt --max_length 64

In [ ]:
# Load all tokenized data
train_code_data = torch.load('train_code.pt')
train_sum_data = torch.load('train_summary.pt')
val_code_data = torch.load('val_code.pt')
val_sum_data = torch.load('val_summary.pt')

# Extract constants from code data (shared tokenizer)
embedding_matrix = train_code_data['embedding_matrix']
pad_id = train_code_data['pad_token_id']
eos_id = train_code_data['eos_token_id']
vocab_size = train_code_data['vocab_size']
embedding_dim = train_code_data['embedding_dim']

print(f'Embedding matrix: {embedding_matrix.shape}')
print(f'Vocab size: {vocab_size}, Embedding dim: {embedding_dim}')
print(f'PAD id: {pad_id}, EOS id: {eos_id}')
print(f'Train samples: {len(train_code_data["token_ids"])}')
print(f'Val samples: {len(val_code_data["token_ids"])}')

## 4. Dataset & DataLoader

In [ ]:
class CodeSumDataset(Dataset):
    def __init__(self, code_ids, sum_ids):
        self.code_ids = code_ids
        self.sum_ids = sum_ids

    def __len__(self):
        return len(self.code_ids)

    def __getitem__(self, idx):
        return (torch.tensor(self.code_ids[idx], dtype=torch.long),
                torch.tensor(self.sum_ids[idx], dtype=torch.long))

def collate_fn(batch):
    codes, sums = zip(*batch)
    codes = pad_sequence(codes, batch_first=True, padding_value=pad_id)
    sums = pad_sequence(sums, batch_first=True, padding_value=pad_id)
    return codes, sums

BATCH_SIZE = 64

train_dataset = CodeSumDataset(train_code_data['token_ids'], train_sum_data['token_ids'])
val_dataset = CodeSumDataset(val_code_data['token_ids'], val_sum_data['token_ids'])

train_loader = DataLoader(train_dataset, BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

## 5. LSTM Encoder-Decoder Model

Architecture (adapted from the class bug-fixing notebook):
- **Encoder:** 2-layer LSTM reads the Java method token embeddings and produces a context (hidden state).
- **Decoder:** 2-layer LSTM takes the encoder's final hidden state and generates natural language summary tokens one at a time.
- **Embeddings:** Shared CodeT5+ pretrained embedding layer (fine-tuned during training), projected from 768d to hidden_dim.
- **Teacher forcing** during training; **greedy autoregressive** decoding at inference.

In [ ]:
HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.2
MAX_GEN_LEN = 64

class Seq2SeqLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_matrix, hidden_dim, num_layers, dropout, pad_id):
        super().__init__()
        embed_dim = embedding_matrix.shape[1]  # 768
        self.pad_id = pad_id

        # Shared pretrained embeddings
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=False, padding_idx=pad_id)
        # Project 768 -> hidden_dim
        self.proj = nn.Linear(embed_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)

        self.encoder = nn.LSTM(hidden_dim, hidden_dim, num_layers,
                               batch_first=True, dropout=dropout)
        self.decoder = nn.LSTM(hidden_dim, hidden_dim, num_layers,
                               batch_first=True, dropout=dropout)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def _embed(self, x):
        return self.dropout(self.proj(self.embedding(x)))

    def forward(self, src, tgt):
        # Encode
        _, hidden = self.encoder(self._embed(src))
        # Decode with teacher forcing: feed tgt[:, :-1], predict tgt[:, 1:]
        dec_out, _ = self.decoder(self._embed(tgt[:, :-1]), hidden)
        return self.fc_out(dec_out)

    def generate(self, src, max_len=MAX_GEN_LEN, eos_id=1):
        """Greedy autoregressive decoding for a single source sequence."""
        self.eval()
        with torch.no_grad():
            _, hidden = self.encoder(self._embed(src))
            # Start token: first token of summary sequences (typically BOS=1 for CodeT5+)
            inp = src.new_ones(1, 1)  # token id 1
            outputs = []
            for _ in range(max_len):
                out, hidden = self.decoder(self._embed(inp), hidden)
                pred = self.fc_out(out).argmax(-1)  # (1,1)
                tok = pred.item()
                if tok == eos_id:
                    break
                outputs.append(tok)
                inp = pred
        return outputs

model = Seq2SeqLSTM(vocab_size, embedding_matrix, HIDDEN_DIM, NUM_LAYERS, DROPOUT, pad_id).to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,} total, '
      f'{sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable')

## 6. Training with Early Stopping (BLEU-1)

- Optimizer: Adam (lr=1e-3)
- Loss: CrossEntropyLoss (ignoring PAD tokens)
- Early stopping: save best checkpoint by validation BLEU-1, patience=3 epochs
- Track train and val loss per epoch for overfitting analysis

In [ ]:
from transformers import AutoTokenizer
import sacrebleu

tokenizer = AutoTokenizer.from_pretrained('Salesforce/codet5p-220m')

def decode_ids(ids):
    """Decode token IDs to text, filtering special tokens."""
    filtered = [i for i in ids if i not in (pad_id, eos_id)]
    return tokenizer.decode(filtered, skip_special_tokens=True).strip()

def compute_val_bleu1(model, val_loader, n_samples=200):
    """Compute BLEU-1 on a subset of validation data."""
    model.eval()
    preds, refs = [], []
    count = 0
    with torch.no_grad():
        for src, tgt in val_loader:
            src, tgt = src.to(DEVICE), tgt.to(DEVICE)
            for i in range(src.size(0)):
                if count >= n_samples:
                    break
                pred_ids = model.generate(src[i:i+1], eos_id=eos_id)
                preds.append(decode_ids(pred_ids))
                refs.append(decode_ids(tgt[i].tolist()))
                count += 1
            if count >= n_samples:
                break
    # sacrebleu BLEU-1
    bleu = sacrebleu.corpus_bleu(preds, [refs], max_ngram_order=1)
    return bleu.score

def compute_val_loss(model, val_loader, criterion):
    model.eval()
    total_loss, total_tokens = 0, 0
    with torch.no_grad():
        for src, tgt in val_loader:
            src, tgt = src.to(DEVICE), tgt.to(DEVICE)
            output = model(src, tgt)
            loss = criterion(output.reshape(-1, output.size(-1)), tgt[:, 1:].reshape(-1))
            total_loss += loss.item() * tgt[:, 1:].numel()
            total_tokens += tgt[:, 1:].numel()
    return total_loss / total_tokens

In [ ]:
NUM_EPOCHS = 15
LR = 1e-3
PATIENCE = 3
CKPT_PATH = 'best_model.pt'

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(ignore_index=pad_id)

train_losses, val_losses, val_bleus = [], [], []
best_bleu = 0
patience_counter = 0

for epoch in range(1, NUM_EPOCHS + 1):
    # --- Train ---
    model.train()
    epoch_loss = 0
    for src, tgt in tqdm(train_loader, desc=f'Epoch {epoch}'):
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        output = model(src, tgt)
        loss = criterion(output.reshape(-1, output.size(-1)), tgt[:, 1:].reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()

    avg_train_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # --- Validate ---
    avg_val_loss = compute_val_loss(model, val_loader, criterion)
    val_losses.append(avg_val_loss)
    bleu1 = compute_val_bleu1(model, val_loader)
    val_bleus.append(bleu1)

    print(f'Epoch {epoch} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val BLEU-1: {bleu1:.2f}')

    # --- Early stopping ---
    if bleu1 > best_bleu:
        best_bleu = bleu1
        patience_counter = 0
        torch.save(model.state_dict(), CKPT_PATH)
        print(f'  -> New best! Saved checkpoint.')
    else:
        patience_counter += 1
        print(f'  -> No improvement. Patience {patience_counter}/{PATIENCE}')
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch}. Best BLEU-1: {best_bleu:.2f}')
            break

print(f'\nBest validation BLEU-1: {best_bleu:.2f}')

## 7. Train vs Validation Loss Analysis (Overfitting Check)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(train_losses) + 1)
ax1.plot(epochs_range, train_losses, 'b-o', label='Train Loss')
ax1.plot(epochs_range, val_losses, 'r-o', label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Train vs Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, val_bleus, 'g-o', label='Val BLEU-1')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('BLEU-1')
ax2.set_title('Validation BLEU-1 (Early Stopping Metric)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Load Best Model & Generate Test Predictions

Load the instructor-provided held-out test set (1,000 Java code-summary pairs) and generate predictions. This test set is used **only once** for final metric reporting.

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()
print('Best model loaded.')

# Load test data from the provided CSV
test_codes, test_refs = [], []
with open('provided-files/dataset/test_dataset_tokenized.csv', 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        test_codes.append(row['code'])
        test_refs.append(row['summary'])

print(f'Test samples: {len(test_codes)}')
print(f'Sample code: {test_codes[0][:150]}...')
print(f'Sample ref:  {test_refs[0]}')

In [ ]:
# Tokenize test code using the same CodeT5+ tokenizer
test_preds = []

for code in tqdm(test_codes, desc='Generating'):
    # Tokenize the code
    ids = tokenizer.encode(code, truncation=True, max_length=256)
    src = torch.tensor([ids], dtype=torch.long).to(DEVICE)
    pred_ids = model.generate(src, eos_id=eos_id)
    pred_text = decode_ids(pred_ids)
    test_preds.append(pred_text)

print(f'Generated {len(test_preds)} predictions')

# Show a few
for i in range(5):
    print(f'\n--- Sample {i} ---')
    print(f'Reference:  {test_refs[i]}')
    print(f'Predicted:  {test_preds[i]}')

## 9. Evaluation Metrics

Compute on the held-out test set:
- **BLEU-1, 2, 3, 4** (sacrebleu)
- **METEOR** (nltk)
- **BERTScore** (bert-score)
- **SIDE** (original implementation)

In [ ]:
import sacrebleu
from nltk.translate.meteor_score import meteor_score as nltk_meteor
from bert_score import score as bert_score_fn

# --- BLEU-1,2,3,4 ---
refs_for_bleu = [test_refs]  # sacrebleu expects list of ref lists
for n in [1, 2, 3, 4]:
    bleu = sacrebleu.corpus_bleu(test_preds, refs_for_bleu, max_ngram_order=n)
    print(f'BLEU-{n}: {bleu.score:.2f}')

# --- METEOR ---
meteor_scores = []
for pred, ref in zip(test_preds, test_refs):
    m = nltk_meteor([ref.split()], pred.split())
    meteor_scores.append(m)
avg_meteor = np.mean(meteor_scores)
print(f'METEOR: {avg_meteor:.4f}')

# --- BERTScore ---
P, R, F1 = bert_score_fn(test_preds, test_refs, lang='en', verbose=True)
print(f'BERTScore (F1): {F1.mean().item():.4f}')

In [ ]:
# --- SIDE ---
# Clone the SIDE repository if not already present
![ -d "SIDE" ] || git clone https://github.com/anthropics/SIDE.git SIDE 2>/dev/null || echo "SIDE repo not found, trying alternative..."

# If the instructor provided a specific SIDE repo URL, update the clone command above.
# For now, we compute SIDE if available, otherwise note it.
try:
    import sys
    sys.path.insert(0, 'SIDE')
    from side import compute_side_score
    side_score = compute_side_score(test_preds, test_refs)
    print(f'SIDE: {side_score:.4f}')
except Exception as e:
    print(f'SIDE computation: {e}')
    print('Note: Update the SIDE repo URL to the one provided by the instructor.')

## 10. Results Summary

In [ ]:
print('=' * 40)
print('FINAL TEST RESULTS')
print('=' * 40)
for n in [1, 2, 3, 4]:
    bleu = sacrebleu.corpus_bleu(test_preds, refs_for_bleu, max_ngram_order=n)
    print(f'BLEU-{n}:      {bleu.score:.2f}')
print(f'METEOR:      {avg_meteor:.4f}')
print(f'BERTScore:   {F1.mean().item():.4f}')
print('=' * 40)